# 05 — Ground an answer in company knowledge
Context is what goes into this run. Knowledge is the document collection
searched for relevant passages. Source IDs make the answer auditable.



## Problem

A fluent answer may omit the company documents or invent vendor claims.


## Concept

CrewAI Knowledge retrieves relevant document chunks using a configured embedder.

Knowledge is retrieved source material, not conversation history. Chunking and embeddings select relevant passages; source IDs let a reviewer connect an output claim back to an input document.


## Abstraction

sample documents → chunks/embeddings → Knowledge retrieval → agent answer. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
company files ──▶ Knowledge sources ──▶ chunks + embedder
                                               │ retrieval
user question ──▶ Crew / Researcher ◀──────────┘
                         └──▶ answer + source IDs + unknowns
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Inspect the document search and explicit embedding configuration

Inspect the document search and explicit embedding configuration.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

import workshop  # configure project-local CrewAI storage
from crewai import Agent, Crew, Process, Task
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

from workshop.common import DATA, embedder_config, model
from workshop.tools import DOCUMENTS, search_documents

print("Transparent search before RAG:", search_documents("private deployment security")[:1])
sources = [
    StringKnowledgeSource(content=f"Source ID: {name}\n{(DATA / name).read_text(encoding='utf-8')}")
    for name in DOCUMENTS
]
embedder = embedder_config()
print("CrewAI Knowledge sources:", len(sources))
print("Embedding provider:", embedder["provider"])



## CrewAI Knowledge retrieval
The embedder is explicit. Changing the chat model alone does not configure
Knowledge or Memory embeddings.



## Step 2 — Attach Knowledge sources to the Crew and demand source IDs and unknowns in the answer

Attach Knowledge sources to the Crew and demand source IDs and unknowns in the answer.


In [ ]:
researcher = Agent(
    role="Company Knowledge Researcher",
    goal="Answer only from named sample company sources",
    backstory="You cite the source ID for every factual claim.",
    llm=model(),
    max_iter=3,
    allow_delegation=False,
)
task = Task(
    description="What is known and unknown about Meridian AI's security? "
    "Cite source IDs. Say when evidence is missing.",
    expected_output="Short answer with source IDs and unknowns.",
    agent=researcher,
)
crew = Crew(
    agents=[researcher], tasks=[task], process=Process.sequential,
    knowledge_sources=sources, embedder=embedder,
)
print(crew.kickoff().raw)


## Execution flow

sample documents → chunks/embeddings → Knowledge retrieval → agent answer. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

Retrieval can return an irrelevant chunk or no chunk. Source IDs in prose are not automatic proof: verify them against the actual documents and explicitly report missing evidence.


## Production implication

Define document ownership, retention, and re-indexing rules. Evaluate retrieval relevance and source attribution separately from answer fluency.


## Your turn

Ask about deletion guarantees and inspect whether the answer admits the gap.

**Checkpoint:** Answer claims name source IDs and identify unverified facts.
